In [17]:
import pandas as pd
from sqlalchemy import create_engine

# Define your connection details
server = 'ctrt-sqlserver.database.windows.net'
database = 'ctrt-sqldb'
username = 'CTRT-admin'
password = 'Hso2Ghana'
table_name = 'bronze.WP_Population_History' 
port = '1433'  # Default is 1433, e.g., '1433'

# Create the connection string for SQLAlchemy
connection_string = (
    f"mssql+pyodbc://{username}:{password}@{server}:{port}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

# Establish the connection using SQLAlchemy create_engine
engine = create_engine(connection_string, echo=False)  # Set echo=True for debugging

# Define the query
query = f"SELECT * FROM {table_name}"

# Execute the query and load the data into a DataFrame
pop_history_df = pd.read_sql(query, engine)


In [18]:
pop_history_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Region      102 non-null    object
 1   Date        102 non-null    object
 2   Population  102 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB


In [19]:
pop_history_df.Population

0       880921
1      2060585
2      1659040
3       901502
4      1301226
        ...   
97            
98      587920
99     1109133
100    6726815
101           
Name: Population, Length: 102, dtype: object

In [20]:
pop_history_df.head(5)

,Region,Date,Population
0,Western North,01/01/2021,880921
1,Western,01/01/2021,2060585
2,Volta,01/01/2021,1659040
3,Upper West,01/01/2021,901502
4,Upper East,01/01/2021,1301226


In [21]:
pop_history_df["Population"]=pd.to_numeric(pop_history_df["Population"],errors="coerce")

In [22]:
pop_history_df["Population"].dtype

dtype('float64')

In [23]:
pop_history_df.head(5)

,Region,Date,Population
0,Western North,01/01/2021,880921.0
1,Western,01/01/2021,2060585.0
2,Volta,01/01/2021,1659040.0
3,Upper West,01/01/2021,901502.0
4,Upper East,01/01/2021,1301226.0


In [24]:
pop_history_df["Date"]=pd.to_datetime(pop_history_df["Date"],errors="coerce")

In [25]:
pop_history_df.head(5)

,Region,Date,Population
0,Western North,2021-01-01,880921.0
1,Western,2021-01-01,2060585.0
2,Volta,2021-01-01,1659040.0
3,Upper West,2021-01-01,901502.0
4,Upper East,2021-01-01,1301226.0


In [26]:
pop_history_df["Month"]=pop_history_df["Date"].dt.month
pop_history_df["Year"]=pop_history_df["Date"].dt.year

In [27]:
pop_history_df.head(5)

,Region,Date,Population,Month,Year
0,Western North,2021-01-01,880921.0,1,2021
1,Western,2021-01-01,2060585.0,1,2021
2,Volta,2021-01-01,1659040.0,1,2021
3,Upper West,2021-01-01,901502.0,1,2021
4,Upper East,2021-01-01,1301226.0,1,2021


In [28]:
pop_history_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Region      102 non-null    object        
 1   Date        102 non-null    datetime64[ns]
 2   Population  72 non-null     float64       
 3   Month       102 non-null    int32         
 4   Year        102 non-null    int32         
dtypes: datetime64[ns](1), float64(1), int32(2), object(1)
memory usage: 3.3+ KB


In [30]:
pop_history_df

,Region,Date,Population,Month,Year
0,Western North,2021-01-01,880921.0,1,2021
1,Western,2021-01-01,2060585.0,1,2021
2,Volta,2021-01-01,1659040.0,1,2021
3,Upper West,2021-01-01,901502.0,1,2021
4,Upper East,2021-01-01,1301226.0,1,2021
...,...,...,...,...,...
97,Bono East,1960-01-01,NaN,1,1960
98,Bono,1960-01-01,587920.0,1,1960
99,Ashanti,1960-01-01,1109133.0,1,1960
100,Ghana,1960-01-01,6726815.0,1,1960


In [31]:
pop_history_df.drop("Month",axis=1,inplace=True)

In [32]:
pop_history_df

,Region,Date,Population,Year
0,Western North,2021-01-01,880921.0,2021
1,Western,2021-01-01,2060585.0,2021
2,Volta,2021-01-01,1659040.0,2021
3,Upper West,2021-01-01,901502.0,2021
4,Upper East,2021-01-01,1301226.0,2021
...,...,...,...,...
97,Bono East,1960-01-01,NaN,1960
98,Bono,1960-01-01,587920.0,1960
99,Ashanti,1960-01-01,1109133.0,1960
100,Ghana,1960-01-01,6726815.0,1960


In [35]:
new_schema = 'silver'
new_table_name = 'MG_Population_History'

# Write DataFrame to the new table in the specified schema in the database
pop_history_df.to_sql(new_table_name, engine, schema=new_schema, if_exists='replace', index=False)

print(f"Data written to {new_schema}.{new_table_name} successfully.")

Data written to silver.MG_Population_History successfully.
